# Laboratorio 21: Optimización de Carteras con VQE

Aplicamos el algoritmo VQE al problema de selección de cartera de Markowitz usando
formulación QUBO y Qiskit 2.x. Compararemos la solución cuántica con la clásica exacta.

**Conceptos clave:** QUBO, Hamiltoniano de Ising, VQE, frontera de Markowitz, aversión al riesgo.

**Prerequisitos:** Módulos 11 (VQE), 12 (Aplicaciones financieras), Lab 28 (VQE-UCCSD).

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from itertools import product as iproduct

from qiskit import QuantumCircuit
from qiskit.circuit import ParameterVector
from qiskit.quantum_info import SparsePauliOp, Statevector
from qiskit.primitives import StatevectorEstimator
from scipy.optimize import minimize

print("Entorno listo — Qiskit 2.x, numpy", np.__version__)

## 1. El problema de Markowitz

Dada una cartera de N activos, buscamos los pesos binarios $x_i \in \{0,1\}$ que:

$$\min_{x} \; q \cdot x^T \Sigma x - \mu^T x$$

donde $\Sigma$ es la matriz de covarianza, $\mu$ el vector de retornos esperados y $q$ la aversión al riesgo.

Para $N=4$ activos, el espacio de búsqueda tiene $2^4=16$ posibles carteras.

In [ ]:
# Datos financieros: 4 activos (ej. acciones tecnológicas)
N = 4
activos = ["AAPL", "MSFT", "GOOGL", "AMZN"]

# Matriz de covarianza anualizada (volatilidades correlacionadas)
cov = np.array([
    [1.00, 0.42, 0.38, 0.35],
    [0.42, 1.00, 0.44, 0.40],
    [0.38, 0.44, 1.00, 0.37],
    [0.35, 0.40, 0.37, 1.00],
]) * 0.04  # escalar a varianzas reales (~20% volatilidad anual)

# Retornos esperados anuales
mu = np.array([0.12, 0.10, 0.11, 0.09])  # 9-12% anual

# Aversión al riesgo
q = 0.5

print("Activos:", activos)
print("Retornos esperados (%):", (mu * 100).round(1))
print("Volatilidades (%):", (np.sqrt(np.diag(cov)) * 100).round(1))

## 2. Formulación QUBO → Hamiltoniano de Ising

Mapeamos el problema QUBO a un operador de Pauli usando la sustitución $x_i = (1 - Z_i)/2$:

$$H = q \sum_{i,j} \Sigma_{ij} \frac{(1-Z_i)(1-Z_j)}{4} - \sum_i \mu_i \frac{1-Z_i}{2}$$

El estado de mínima energía de $H$ corresponde a la cartera óptima.

In [ ]:
def build_portfolio_hamiltonian(cov, mu, q, n):
    """Construye el Hamiltoniano de Ising para el problema de cartera."""
    pauli_list = []
    
    # Término cuadrático: q * Σ_ij * (I - Zi)(I - Zj)/4
    for i in range(n):
        for j in range(n):
            coeff_ij = q * cov[i, j] / 4.0
            # Término constante (II): coeff_ij * 1
            pauli_list.append(("I" * n, coeff_ij))
            # Término Zi: -coeff_ij
            zi = "I" * (n - 1 - i) + "Z" + "I" * i
            pauli_list.append((zi, -coeff_ij))
            # Término Zj: -coeff_ij (solo si i != j)
            if i != j:
                zj = "I" * (n - 1 - j) + "Z" + "I" * j
                pauli_list.append((zj, -coeff_ij))
            # Término ZiZj: coeff_ij
            if i != j:
                zizj = ["I"] * n
                zizj[n - 1 - i] = "Z"
                zizj[n - 1 - j] = "Z"
                pauli_list.append(("".join(zizj), coeff_ij))
    
    # Término lineal: -mu_i * (I - Zi)/2
    for i in range(n):
        pauli_list.append(("I" * n, -mu[i] / 2.0))
        zi = "I" * (n - 1 - i) + "Z" + "I" * i
        pauli_list.append((zi, mu[i] / 2.0))
    
    return SparsePauliOp.from_list(pauli_list).simplify()

H = build_portfolio_hamiltonian(cov, mu, q, N)
print(f"Hamiltoniano H: {len(H)} términos de Pauli")
print("Primeros 5 términos:")
for op, coeff in zip(H.paulis[:5], H.coeffs[:5]):
    print(f"  {op}: {coeff:.4f}")

## 3. Solución clásica exacta (referencia)

Con $N=4$ podemos enumerar los $2^4=16$ posibles portafolios y encontrar el óptimo exacto.
Esto nos dará la referencia para evaluar VQE.

In [ ]:
# Enumeración exhaustiva (factible para N=4)
mejor_cartera = None
mejor_valor = np.inf

resultados = []
for bits in iproduct([0, 1], repeat=N):
    x = np.array(bits)
    valor = q * x @ cov @ x - mu @ x
    resultados.append((bits, valor))
    if valor < mejor_valor:
        mejor_valor = valor
        mejor_cartera = bits

resultados.sort(key=lambda r: r[1])

print("Top 5 carteras (menor costo = mejor):")
print(f"{'Cartera':>20}  {'Riesgo':>8}  {'Retorno':>8}  {'Costo':>10}")
print("-" * 55)
for bits, val in resultados[:5]:
    x = np.array(bits)
    riesgo = np.sqrt(x @ cov @ x) * 100
    retorno = mu @ x * 100
    seleccion = [activos[i] for i in range(N) if bits[i]]
    print(f"{str(seleccion):>20}  {riesgo:>7.1f}%  {retorno:>7.1f}%  {val:>10.4f}")

print(f"\nCartera óptima exacta: {[activos[i] for i in range(N) if mejor_cartera[i]]}")
print(f"Costo óptimo: {mejor_valor:.4f}")

## 4. Ansatz y VQE

Usamos un ansatz de circuito variacional de 2 capas (RyRz + CNOT), que tiene suficiente
expresividad para explorar el espacio de $2^4$ estados de base computacional.

In [ ]:
def build_ansatz(n_qubits, reps=2):
    """Ansatz RyRz + CNOT en cadena linear."""
    params = ParameterVector("θ", 2 * n_qubits * reps + 2 * n_qubits)
    qc = QuantumCircuit(n_qubits)
    idx = 0
    for _ in range(reps):
        for i in range(n_qubits):
            qc.ry(params[idx], i); idx += 1
            qc.rz(params[idx], i); idx += 1
        for i in range(n_qubits - 1):
            qc.cx(i, i + 1)
    # Capa final
    for i in range(n_qubits):
        qc.ry(params[idx], i); idx += 1
        qc.rz(params[idx], i); idx += 1
    return qc, params

ansatz, params = build_ansatz(N, reps=2)
print(f"Ansatz: {ansatz.num_parameters} parámetros, profundidad {ansatz.depth()}")
ansatz.draw("text")

In [ ]:
# VQE con StatevectorEstimator (simulación exacta)
estimator = StatevectorEstimator()
energy_history = []

def cost(theta):
    job = estimator.run([(ansatz, H, [theta])])
    E = float(job.result()[0].data.evs)
    energy_history.append(E)
    return E

np.random.seed(42)
theta0 = np.random.uniform(-np.pi, np.pi, ansatz.num_parameters)

print("Optimizando con COBYLA...")
result = minimize(cost, theta0, method="COBYLA",
                  options={"maxiter": 800, "rhobeg": 0.5})

print(f"\nEnergía VQE: {result.fun:.4f}")
print(f"Energía exacta: {mejor_valor:.4f}")
print(f"Error: {abs(result.fun - mejor_valor):.4f}")
print(f"Iteraciones: {len(energy_history)}")

In [ ]:
# Identificar la cartera que VQE encontró
theta_opt = result.x
sv = Statevector(ansatz.assign_parameters(theta_opt))
probs = sv.probabilities_dict()

# Top 3 estados por probabilidad
top_states = sorted(probs.items(), key=lambda x: x[1], reverse=True)[:5]

print("Estados más probables tras VQE:")
print(f"{'Estado':>10}  {'Prob':>6}  {'Cartera':>25}  {'Costo':>8}")
print("-" * 60)
for state, prob in top_states:
    bits = tuple(int(b) for b in reversed(state))
    seleccion = [activos[i] for i in range(N) if bits[i]]
    x = np.array(bits)
    costo = q * x @ cov @ x - mu @ x
    print(f"{state:>10}  {prob:>6.3f}  {str(seleccion):>25}  {costo:>8.4f}")

# Verificar si encontró el óptimo
estado_opt = "".join(str(b) for b in reversed(list(mejor_cartera)))
print(f"\n¿VQE encontró el óptimo ({estado_opt})? {'✅' if top_states[0][0] == estado_opt else '❌ (ver estado más probable arriba)'}")

In [ ]:
# Visualización: convergencia y frontera de Markowitz
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Panel 1: convergencia de energía
axes[0].plot(energy_history, alpha=0.8, color='steelblue')
axes[0].axhline(mejor_valor, color='red', ls='--', lw=2, label=f'Óptimo exacto = {mejor_valor:.3f}')
axes[0].set_xlabel("Iteración"); axes[0].set_ylabel("Energía (costo)")
axes[0].set_title("Convergencia de VQE — Optimización de Cartera")
axes[0].legend(); axes[0].grid(True, alpha=0.3)

# Panel 2: frontera riesgo-retorno de las 16 carteras
riesgos  = [np.sqrt(np.array(b) @ cov @ np.array(b)) * 100 for b,_ in resultados]
retornos = [mu @ np.array(b) * 100 for b,_ in resultados]
costos   = [v for _,v in resultados]

sc = axes[1].scatter(riesgos, retornos, c=costos, cmap='RdYlGn_r', s=80, zorder=3)
plt.colorbar(sc, ax=axes[1], label='Costo QUBO')

# Destacar el óptimo
r_opt = np.sqrt(np.array(mejor_cartera) @ cov @ np.array(mejor_cartera)) * 100
ret_opt = mu @ np.array(mejor_cartera) * 100
axes[1].scatter([r_opt], [ret_opt], marker='*', s=250, color='gold', zorder=5,
                label=f'Óptimo: {[activos[i] for i in range(N) if mejor_cartera[i]]}')
axes[1].set_xlabel("Riesgo (volatilidad %)"); axes[1].set_ylabel("Retorno esperado (%)")
axes[1].set_title("Espacio riesgo-retorno (16 carteras posibles)")
axes[1].legend(fontsize=9); axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig("cartera_vqe.png", dpi=100, bbox_inches='tight')
plt.show()
print("Figura guardada en cartera_vqe.png")